# NOURA EL KHOLTI

## Classifieur Naive Bayes

Le classifieur Naive Bayes (Bayésien Naïf) est un modèle probabiliste basé sur le théorème de Bayes. Il est qualifié de "naïf" car il suppose que toutes les caractéristiques (features) sont indépendantes les unes des autres.

### 1. Formule de Bayes

$$P(Classe | Caractéristiques) \propto P(Classe) \times \prod_{i=1}^{n} P(Caractéristique_i | Classe)$$

Le processus consiste à calculer deux types de probabilités:
 1. Les Probabilités a priori : $P(Classe)$; La probabilité globale de chaque classe dans le jeu de données.
 2. Les Vraisemblances : $P(Caractéristique | Classe)$; La probabilité d'observer une valeur spécifique sachant la classe.

### 2. Préparation des données

Nous utilisons un jeu de données simulant des consultations médicales où l'on évalue trois symptômes : la fièvre, la toux et la fatigue.

In [ ]:
import pandas as pd

# Données d'apprentissage : Symptômes et Diagnostics
donnees_patients = {
    'Fievre': ['Forte', 'Aucune', 'Legere', 'Forte', 'Aucune', 'Legere', 'Legere', 'Forte', 'Aucune', 'Legere'],
    'Toux': ['Oui', 'Oui', 'Non', 'Oui', 'Non', 'Oui', 'Non', 'Oui', 'Oui', 'Non'],
    'Fatigue': ['Oui', 'Non', 'Oui', 'Oui', 'Non', 'Non', 'Oui', 'Non', 'Oui', 'Non'],
    'Diagnostic': ['Grippe', 'Rhume', 'Allergie', 'Grippe', 'Rhume', 'Allergie', 'Allergie', 'Grippe', 'Rhume', 'Allergie']
}

df = pd.DataFrame(donnees_patients)
print("Base de données des patients :")
print(df)

```
Base de données des patients :
   Fievre Toux Fatigue Diagnostic
0   Forte  Oui     Oui     Grippe
1  Aucune  Oui     Non      Rhume
2  Legere  Non     Oui   Allergie
3   Forte  Oui     Oui     Grippe
4  Aucune  Non     Non      Rhume
5  Legere  Oui     Non   Allergie
6  Legere  Non     Oui   Allergie
7   Forte  Oui     Non     Grippe
8  Aucune  Oui     Oui      Rhume
9  Legere  Non     Non   Allergie
```

### 3. Entraînement du Modèle

Cette étape consiste à extraire les connaissances (probabilités) du jeu de données.

In [ ]:
def entrainer_modele_medical(donnees, cible):
    # 1. Calcul des probabilités a priori P(Maladie)
    total_cas = len(donnees)
    classes = donnees[cible].unique()
    prieurs = {m: len(donnees[donnees[cible] == m]) / total_cas for m in classes}
    
    # 2. Calcul des vraisemblances P(Symptôme|Maladie)
    vraisemblances = {}
    symptomes = donnees.columns.drop(cible)
    
    for s in symptomes:
        vraisemblances[s] = {}
        etats_possibles = donnees[s].unique()
        
        for etat in etats_possibles:
            vraisemblances[s][etat] = {}
            for m in classes:
                subset = donnees[donnees[cible] == m]
                # Fréquence du symptôme pour cette maladie spécifique
                nb_occurences = len(subset[subset[s] == etat])
                vraisemblances[s][etat][m] = nb_occurences / len(subset)
                
    return prieurs, vraisemblances

prieurs, vraisemblances = entrainer_modele_medical(df, 'Diagnostic')
print("Le modèle a été entraîné sur les données cliniques.")

```
Le modèle a été entraîné sur les données cliniques.
```

### 4. Prédiction et Scoring

Cette fonction évalue les symptômes d'un nouveau patient pour estimer les probabilités de chaque diagnostic.

In [ ]:
def diagnostiquer_patient(symptomes_observes, prieurs, vraisemblances):
    scores = {}
    
    for m in prieurs:
        # Score initial basé sur la fréquence de la maladie (Prior)
        scores[m] = prieurs[m]
        
        # Ajustement du score selon chaque symptôme observé
        for symp, valeur in symptomes_observes.items():
            if valeur in vraisemblances[symp]:
                scores[m] *= vraisemblances[symp][valeur][m]
            else:
                # Utilisation d'une constante très faible pour éviter le score nul
                scores[m] *= 1e-9 
                
    # Normalisation pour obtenir des probabilités en pourcentage
    total_score = sum(scores.values())
    probabilites = {m: (scores[m] / total_score) * 100 for m in scores}
    
    # Sélection du diagnostic le plus probable
    diagnostic_final = max(probabilites, key=probabilites.get)
    
    return diagnostic_final, probabilites

### 5. Lancement et Test

Simulation d'une nouvelle consultation médicale.

In [ ]:
# Nouveau cas clinique : Le patient a une toux persistante mais pas de fièvre ni de fatigue.
patient_X = {
    'Fievre': 'Aucune',
    'Toux': 'Oui',
    'Fatigue': 'Non'
}

resultat, details_probas = diagnostiquer_patient(patient_X, prieurs, vraisemblances)

print("--- RÉSULTATS DU DIAGNOSTIC ---")
print(f"Symptômes rapportés : {patient_X}")
print("-" * 30)
for maladie, proba in details_probas.items():
    print(f"Probabilité {maladie} : {proba:.2f}%")
print("-" * 30)
print(f"Conclusion clinique : Le patient semble souffrir d'un(e) {resultat.upper()}.")

```
--- RÉSULTATS DU DIAGNOSTIC ---
Symptômes rapportés : {'Fievre': 'Aucune', 'Toux': 'Oui', 'Fatigue': 'Non'}
------------------------------
Probabilité Grippe : 0.00%
Probabilité Rhume : 100.00%
Probabilité Allergie : 0.00%
------------------------------
Conclusion clinique : Le patient semble souffrir d'un(e) RHUME.
```